# Notebook 5 – Penanganan Class Imbalance

**Tujuan:** Mengatasi ketidakseimbangan kelas ekstrem pada dataset kontrak Bridge
(rasio 243×) menggunakan empat teknik berbeda dan membandingkan dampaknya
terhadap F1 Macro.

**Teknik yang diuji:**
1. Random Oversampling (`RandomOverSampler`)
2. SMOTE (`Synthetic Minority Over-sampling Technique`)
3. Custom Class Weights (inverse-frequency)
4. Threshold Calibration per kelas

**Output:**
- Visualisasi & CSV di `experiments/2026-07-01/results/`
- Model eksperimental di `experiments/2026-07-01/models/`

> **Prasyarat:** `data/processed/` dan `outputs/models/*.pkl` harus sudah ada
> (dari `01_dataset_analysis.ipynb` dan `03_modeling.ipynb`).

---
## 0. Setup

In [ ]:
import sys
import warnings
import pickle
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.metrics import f1_score, accuracy_score

warnings.filterwarnings('ignore')

ROOT    = Path('..').resolve()
EXP_DIR = ROOT / 'experiments' / '2026-07-01'
OUT     = EXP_DIR / 'results'
MDL_DIR = EXP_DIR / 'models'
OUT.mkdir(parents=True, exist_ok=True)
MDL_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(ROOT))

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})
sns.set_palette('tab10')
MODEL_COLORS = ['#4472C4', '#ED7D31', '#70AD47', '#FF6B6B', '#A855F7']

print(f'ROOT      : {ROOT}')
print(f'Output dir: {OUT}')
print('Setup selesai.')

In [ ]:
from src.preprocessing import load_splits
from src.evaluation import evaluate, compare_models

df_train, df_val, df_test, feature_cols, le = load_splits(ROOT / 'data' / 'processed')

X_train = df_train[feature_cols].values.astype(np.float32)
y_train = df_train['label'].values
X_val   = df_val[feature_cols].values.astype(np.float32)
y_val   = df_val['label'].values
X_test  = df_test[feature_cols].values.astype(np.float32)
y_test  = df_test['label'].values

n_classes  = len(le.classes_)
class_names = list(le.classes_)

print(f'Train : {X_train.shape}')
print(f'Val   : {X_val.shape}')
print(f'Test  : {X_test.shape}')
print(f'Kelas : {n_classes}')

---
## 1. Analisis Class Imbalance

In [ ]:
# Hitung distribusi kelas pada training set
train_cnt = Counter(y_train)
counts    = np.array([train_cnt.get(i, 0) for i in range(n_classes)])
labels    = np.array(class_names)

# Statistik imbalance
sort_idx   = np.argsort(counts)[::-1]
max_cls    = labels[sort_idx[0]]
min_cls    = labels[counts > 0][np.argmin(counts[counts > 0])]
ratio      = counts.max() / counts[counts > 0].min()
n_rare     = (counts < 20).sum()
n_rare50   = (counts < 50).sum()
n_zero     = (counts == 0).sum()

print('=' * 55)
print('  STATISTIK CLASS IMBALANCE – Training Set')
print('=' * 55)
print(f'  Total kelas           : {n_classes}')
print(f'  Kelas terbesar        : {max_cls} ({counts.max():,} sampel)')
print(f'  Kelas terkecil (aktif): {min_cls} ({counts[counts > 0].min():,} sampel)')
print(f'  Rasio imbalance       : {ratio:.1f}×')
print(f'  Kelas dengan < 20 sampel : {n_rare}')
print(f'  Kelas dengan < 50 sampel : {n_rare50}')
print(f'  Kelas tidak ada di train : {n_zero}')
print('=' * 55)
print()
print('Distribusi kelas (urut terbanyak → terkecil):')
print(f'  {"Kontrak":<8} {"Train":>8} {"Val":>8} {"Test":>8} {"% Train":>8}')
print('  ' + '─' * 42)
val_cnt  = Counter(y_val)
test_cnt = Counter(y_test)
for idx in sort_idx:
    lbl = labels[idx]
    tr  = train_cnt.get(idx, 0)
    vl  = val_cnt.get(idx, 0)
    ts  = test_cnt.get(idx, 0)
    pct = tr / len(y_train) * 100
    flag = ' ⚠' if tr < 20 else ''
    print(f'  {lbl:<8} {tr:>8,} {vl:>8,} {ts:>8,} {pct:>7.1f}%{flag}')

print()
print('  ⚠ = kelas langka (< 20 sampel di training)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ── Distribusi log-scale ──────────────────────────────────────────
ax = axes[0]
sorted_counts = counts[sort_idx]
sorted_labels = labels[sort_idx]
bar_colors = ['#C62828' if c < 20 else '#FF6B35' if c < 50 else '#1565C0'
              for c in sorted_counts]
bars = ax.bar(range(n_classes), sorted_counts, color=bar_colors,
              edgecolor='white', linewidth=0.8)
ax.set_xticks(range(n_classes))
ax.set_xticklabels(sorted_labels, rotation=90, fontsize=8)
ax.set_yscale('log')
ax.set_title('Distribusi Kelas Training Set (log scale)', fontweight='bold')
ax.set_ylabel('Jumlah Sampel (log)')
ax.set_xlabel('Kontrak')
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color='#C62828', label='Sangat langka (< 20)'),
    Patch(color='#FF6B35', label='Langka (20–49)'),
    Patch(color='#1565C0', label='Normal (≥ 50)'),
], fontsize=9)

# ── Pie: kategori kontrak ─────────────────────────────────────────
ax = axes[1]
from src.features import get_contract_category
cat_buckets = {'Sangat langka\n(< 20)': 0, 'Langka\n(20–49)': 0,
               'Sedang\n(50–199)': 0, 'Banyak\n(≥ 200)': 0}
for c in counts:
    if c < 20:   cat_buckets['Sangat langka\n(< 20)'] += 1
    elif c < 50: cat_buckets['Langka\n(20–49)'] += 1
    elif c < 200:cat_buckets['Sedang\n(50–199)'] += 1
    else:        cat_buckets['Banyak\n(≥ 200)'] += 1

pie_colors = ['#C62828', '#FF6B35', '#FFC107', '#1565C0']
wedges, texts, autotexts = ax.pie(
    cat_buckets.values(),
    labels=cat_buckets.keys(),
    colors=pie_colors,
    autopct=lambda p: f'{p:.0f}%\n({int(round(p*n_classes/100))} kelas)',
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    textprops={'fontsize': 9},
)
ax.set_title(f'Kategori Kelangkaan Kelas\n(total {n_classes} kelas)', fontweight='bold')

plt.suptitle(
    f'Class Imbalance – Rasio {ratio:.0f}× ({class_names[sort_idx[0]]} vs {min_cls})',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig(OUT / 'nb05_class_distribution.png', bbox_inches='tight', dpi=150)
plt.show()
print('Gambar disimpan.')

---
## 2. Baseline – Model Tanpa Teknik Imbalance

Gunakan XGBoost dan LightGBM (best performers dari notebook 03) sebagai baseline.

In [ ]:
from src.models import XGBModel, LGBMModel

# Load model yang sudah dilatih
baseline_models = {}
for name in ['XGBoost', 'LightGBM']:
    path = ROOT / 'outputs' / 'models' / f'{name.lower()}.pkl'
    assert path.exists(), f'Model tidak ditemukan: {path}. Jalankan 03_modeling.ipynb.'
    with open(path, 'rb') as f:
        baseline_models[name] = pickle.load(f)
    print(f'  Loaded: {name}')

# Evaluasi baseline pada validation set
baseline_results = {}
for name, model in baseline_models.items():
    y_pred  = model.predict(X_val)
    y_proba = model.predict_proba(X_val)
    res = evaluate(y_val, y_pred, y_proba, le, model_name=f'{name} (Baseline)')
    baseline_results[name] = res
    print(f'  {name}: Acc={res["accuracy"]:.4f}  F1m={res["f1_macro"]:.4f}  '
          f'F1w={res["f1_weighted"]:.4f}  Top3={res.get("top_3_accuracy",0):.4f}')

---
## 3. Teknik 1 – Random Oversampling

Duplikasi sampel secara acak pada kelas minoritas hingga semua kelas
memiliki jumlah sampel yang seimbang. Sederhana dan aman untuk kelas
dengan sampel sangat sedikit (< 6 sampel).

In [ ]:
import time
from imblearn.over_sampling import RandomOverSampler

print('Menerapkan RandomOverSampler ...')
ros = RandomOverSampler(random_state=42)
X_ros, y_ros = ros.fit_resample(X_train, y_train)

ros_cnt = Counter(y_ros)
print(f'  Train sebelum : {len(y_train):,} sampel  |  kelas terbesar: {max(Counter(y_train).values()):,}')
print(f'  Train sesudah : {len(y_ros):,} sampel  |  kelas terbesar: {max(ros_cnt.values()):,}')
print(f'  Rasio baru    : 1.0× (semua kelas seimbang)')

ros_results = {}
for name, ModelCls in [('XGBoost', XGBModel), ('LightGBM', LGBMModel)]:
    model = ModelCls()
    print(f'\n[TRAIN] {name} + RandomOverSampler ...', end=' ')
    t0 = time.time()
    model.fit(X_ros, y_ros)
    elapsed = time.time() - t0
    print(f'{elapsed:.1f}s')

    y_pred  = model.predict(X_val)
    y_proba = model.predict_proba(X_val)
    res = evaluate(y_val, y_pred, y_proba, le, model_name=f'{name} + ROS')
    res['train_time'] = elapsed
    ros_results[name] = res

    bl = baseline_results[name]
    delta_acc = res['accuracy']  - bl['accuracy']
    delta_f1m = res['f1_macro']  - bl['f1_macro']
    delta_f1w = res['f1_weighted'] - bl['f1_weighted']
    sign = lambda v: f'+{v:.4f}' if v >= 0 else f'{v:.4f}'
    print(f'  Acc={res["accuracy"]:.4f} ({sign(delta_acc)})  '
          f'F1m={res["f1_macro"]:.4f} ({sign(delta_f1m)})  '
          f'F1w={res["f1_weighted"]:.4f} ({sign(delta_f1w)})')

    model.save(MDL_DIR / f'{name.lower()}_ros.pkl')

---
## 4. Teknik 2 – SMOTE

Bangkitkan sampel sintetis pada kelas minoritas berdasarkan interpolasi
antara tetangga terdekat. Membutuhkan minimal `k_neighbors + 1` sampel
per kelas — kelas dengan sampel sangat sedikit (< 6) disampling random
terlebih dahulu.

In [ ]:
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler as _ROS

# SMOTE butuh minimal k+1 sampel per kelas.
# Kelas dengan < k+1 sampel perlu di-oversample dulu.
k_neighbors = 5
min_for_smote = k_neighbors + 1  # = 6

rare_classes = {cls: cnt for cls, cnt in Counter(y_train).items()
                if cnt < min_for_smote}
print(f'Kelas dengan < {min_for_smote} sampel (pre-sample sebelum SMOTE): '
      f'{len(rare_classes)} kelas')

# Pre-oversample kelas sangat langka ke minimal min_for_smote
pre_strategy = {cls: min_for_smote for cls, cnt in Counter(y_train).items()
                if cnt < min_for_smote}
if pre_strategy:
    pre_ros = _ROS(sampling_strategy=pre_strategy, random_state=42)
    X_pre, y_pre = pre_ros.fit_resample(X_train, y_train)
    print(f'  Setelah pre-sample: {len(y_pre):,} sampel')
else:
    X_pre, y_pre = X_train, y_train

# Terapkan SMOTE
print(f'Menerapkan SMOTE (k_neighbors={k_neighbors}) ...')
smote = SMOTE(k_neighbors=k_neighbors, random_state=42)
X_smote, y_smote = smote.fit_resample(X_pre, y_pre)

smote_cnt = Counter(y_smote)
print(f'  Train sebelum : {len(y_train):,} sampel')
print(f'  Train sesudah : {len(y_smote):,} sampel')
print(f'  Rasio baru    : 1.0×')

smote_results = {}
for name, ModelCls in [('XGBoost', XGBModel), ('LightGBM', LGBMModel)]:
    model = ModelCls()
    print(f'\n[TRAIN] {name} + SMOTE ...', end=' ')
    t0 = time.time()
    model.fit(X_smote, y_smote)
    elapsed = time.time() - t0
    print(f'{elapsed:.1f}s')

    y_pred  = model.predict(X_val)
    y_proba = model.predict_proba(X_val)
    res = evaluate(y_val, y_pred, y_proba, le, model_name=f'{name} + SMOTE')
    res['train_time'] = elapsed
    smote_results[name] = res

    bl = baseline_results[name]
    sign = lambda v: f'+{v:.4f}' if v >= 0 else f'{v:.4f}'
    print(f'  Acc={res["accuracy"]:.4f} ({sign(res["accuracy"]-bl["accuracy"])})  '
          f'F1m={res["f1_macro"]:.4f} ({sign(res["f1_macro"]-bl["f1_macro"])})')

    model.save(MDL_DIR / f'{name.lower()}_smote.pkl')

---
## 5. Teknik 3 – Custom Class Weights

Alih-alih oversampling, berikan bobot berbeda pada kelas saat training:
- `balanced`: bobot = n_total / (n_classes × n_i)
- `sqrt_inv`: bobot = 1 / sqrt(n_i) (lebih moderat)
- `log_inv`:  bobot = 1 / log(1 + n_i) (paling moderat)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

def make_sample_weights(y: np.ndarray, scheme: str) -> np.ndarray:
    counts_arr = np.bincount(y, minlength=n_classes).astype(float)
    counts_arr[counts_arr == 0] = 1.0  # avoid div-by-zero
    if scheme == 'balanced':
        w = len(y) / (n_classes * counts_arr)
    elif scheme == 'sqrt_inv':
        w = 1.0 / np.sqrt(counts_arr)
    elif scheme == 'log_inv':
        w = 1.0 / np.log1p(counts_arr)
    else:
        raise ValueError(scheme)
    w /= w.mean()  # normalize so mean weight = 1
    return w[y]    # per-sample weights

weight_schemes = ['balanced', 'sqrt_inv', 'log_inv']

weight_results = {}  # key = f'{model}_{scheme}'
for name, ModelCls in [('XGBoost', XGBModel), ('LightGBM', LGBMModel)]:
    for scheme in weight_schemes:
        sw = make_sample_weights(y_train, scheme)
        model = ModelCls()
        print(f'[TRAIN] {name} + weights={scheme} ...', end=' ')
        t0 = time.time()
        model.fit(X_train, y_train, sample_weight=sw)
        elapsed = time.time() - t0
        print(f'{elapsed:.1f}s')

        y_pred  = model.predict(X_val)
        y_proba = model.predict_proba(X_val)
        res = evaluate(y_val, y_pred, y_proba, le,
                       model_name=f'{name} + {scheme}')
        res['train_time'] = elapsed
        weight_results[f'{name}_{scheme}'] = res

        bl = baseline_results[name]
        sign = lambda v: f'+{v:.4f}' if v >= 0 else f'{v:.4f}'
        print(f'  Acc={res["accuracy"]:.4f} ({sign(res["accuracy"]-bl["accuracy"])})  '
              f'F1m={res["f1_macro"]:.4f} ({sign(res["f1_macro"]-bl["f1_macro"])})')

---
## 6. Teknik 4 – Threshold Calibration per Kelas

Baseline model menggunakan argmax probabilitas. Dengan mengubah threshold
per kelas, kita dapat meningkatkan recall kelas minoritas pada biaya
presisi yang lebih rendah. Threshold dioptimalkan pada validation set.

In [ ]:
def calibrate_thresholds(
    y_true: np.ndarray,
    y_proba: np.ndarray,
    thresholds: np.ndarray = np.linspace(0.1, 0.9, 17),
) -> np.ndarray:
    """Find per-class threshold that maximises F1 on a given set."""
    n_cls    = y_proba.shape[1]
    best_thr = np.full(n_cls, 0.5)
    for cls_idx in range(n_cls):
        y_bin = (y_true == cls_idx).astype(int)
        if y_bin.sum() == 0:
            continue
        best_f1 = -1.0
        for thr in thresholds:
            y_pred_bin = (y_proba[:, cls_idx] >= thr).astype(int)
            f1 = f1_score(y_bin, y_pred_bin, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_thr[cls_idx] = thr
    return best_thr


def predict_with_thresholds(
    y_proba: np.ndarray,
    thresholds: np.ndarray,
) -> np.ndarray:
    """Predict class by picking the class whose calibrated-prob is highest."""
    calibrated = y_proba / (thresholds + 1e-9)
    return calibrated.argmax(axis=1)


threshold_results = {}
for name, model in baseline_models.items():
    y_proba_val  = model.predict_proba(X_val)
    y_proba_test = model.predict_proba(X_test)

    print(f'Kalibrasi threshold untuk {name} (menggunakan val set) ...')
    thresholds = calibrate_thresholds(y_val, y_proba_val)

    # Evaluasi pada validation
    y_pred_cal_val = predict_with_thresholds(y_proba_val, thresholds)
    res_val = evaluate(y_val, y_pred_cal_val, y_proba_val, le,
                       model_name=f'{name} + Threshold (Val)')
    bl = baseline_results[name]
    sign = lambda v: f'+{v:.4f}' if v >= 0 else f'{v:.4f}'
    print(f'  Val  Acc={res_val["accuracy"]:.4f} ({sign(res_val["accuracy"]-bl["accuracy"])})  '
          f'F1m={res_val["f1_macro"]:.4f} ({sign(res_val["f1_macro"]-bl["f1_macro"])})')

    # Evaluasi pada test (perkiraan generalisasi)
    y_pred_cal_test = predict_with_thresholds(y_proba_test, thresholds)
    res_test = evaluate(y_test, y_pred_cal_test, y_proba_test, le,
                        model_name=f'{name} + Threshold (Test)')
    print(f'  Test Acc={res_test["accuracy"]:.4f} ({sign(res_test["accuracy"]-bl["accuracy"])})  '
          f'F1m={res_test["f1_macro"]:.4f} ({sign(res_test["f1_macro"]-bl["f1_macro"])})')

    threshold_results[name] = {'val': res_val, 'test': res_test, 'thresholds': thresholds}
    np.save(MDL_DIR / f'{name.lower()}_thresholds.npy', thresholds)
    print(f'  Threshold disimpan: {name.lower()}_thresholds.npy')

---
## 7. Perbandingan Semua Teknik

In [ ]:
# Kumpulkan semua hasil dalam satu tabel
rows = []
for name in ['XGBoost', 'LightGBM']:
    # Baseline
    bl = baseline_results[name]
    rows.append({'Teknik': f'{name} – Baseline', 'Model': name,
                 'Accuracy': bl['accuracy'], 'F1 Macro': bl['f1_macro'],
                 'F1 Weighted': bl['f1_weighted'],
                 'Top-3': bl.get('top_3_accuracy', 0)})
    # ROS
    r = ros_results[name]
    rows.append({'Teknik': f'{name} – RandomOversample', 'Model': name,
                 'Accuracy': r['accuracy'], 'F1 Macro': r['f1_macro'],
                 'F1 Weighted': r['f1_weighted'],
                 'Top-3': r.get('top_3_accuracy', 0)})
    # SMOTE
    r = smote_results[name]
    rows.append({'Teknik': f'{name} – SMOTE', 'Model': name,
                 'Accuracy': r['accuracy'], 'F1 Macro': r['f1_macro'],
                 'F1 Weighted': r['f1_weighted'],
                 'Top-3': r.get('top_3_accuracy', 0)})
    # Best weight scheme
    best_w = max(
        [weight_results[f'{name}_{s}'] for s in weight_schemes],
        key=lambda r: r['f1_macro']
    )
    rows.append({'Teknik': f'{name} – {best_w["model"].split("+")[-1].strip()} (best weight)',
                 'Model': name,
                 'Accuracy': best_w['accuracy'], 'F1 Macro': best_w['f1_macro'],
                 'F1 Weighted': best_w['f1_weighted'],
                 'Top-3': best_w.get('top_3_accuracy', 0)})
    # Threshold calibration
    r = threshold_results[name]['val']
    rows.append({'Teknik': f'{name} – Threshold Calib', 'Model': name,
                 'Accuracy': r['accuracy'], 'F1 Macro': r['f1_macro'],
                 'F1 Weighted': r['f1_weighted'],
                 'Top-3': r.get('top_3_accuracy', 0)})

cmp_df = pd.DataFrame(rows).set_index('Teknik')
cmp_df.to_csv(OUT / 'nb05_comparison.csv')

print('=== Perbandingan Teknik Penanganan Imbalance (Validation Set) ===')
print()
print(f'  {"Teknik":<45} {"Acc":>8} {"F1 Mac":>8} {"F1 Wt":>8} {"Top-3":>7}')
print('  ' + '─' * 80)
for i, row in enumerate(rows):
    if i > 0 and rows[i]['Model'] != rows[i-1]['Model']:
        print()
    t = row['Teknik']
    is_baseline = 'Baseline' in t
    prefix = '→ ' if not is_baseline else '   '
    print(f'  {prefix}{t:<43} {row["Accuracy"]:>8.4f} {row["F1 Macro"]:>8.4f}'
          f' {row["F1 Weighted"]:>8.4f} {row["Top-3"]:>7.4f}')

print()
best_f1m = max(rows, key=lambda r: r['F1 Macro'])
best_acc = max(rows, key=lambda r: r['Accuracy'])
print(f'  F1 Macro terbaik: {best_f1m["Teknik"]} → {best_f1m["F1 Macro"]:.4f}')
print(f'  Accuracy terbaik: {best_acc["Teknik"]} → {best_acc["Accuracy"]:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 7))

metrics_to_plot = ['F1 Macro', 'Accuracy']
titles          = ['F1 Macro (target utama)', 'Accuracy']

# Pisahkan baris per model
for ax, metric, title in zip(axes, metrics_to_plot, titles):
    for model_name, color in [('XGBoost', '#ED7D31'), ('LightGBM', '#70AD47')]:
        subset  = [r for r in rows if r['Model'] == model_name]
        labels  = [r['Teknik'].replace(f'{model_name} – ', '') for r in subset]
        values  = [r[metric] for r in subset]
        x       = np.arange(len(labels))

        if model_name == 'XGBoost':
            bars = ax.bar(x - 0.2, values, 0.38, label=model_name,
                          color=color, alpha=0.85, edgecolor='white')
            ax.bar_label(bars, fmt='%.4f', fontsize=8, rotation=90, padding=4)
        else:
            bars = ax.bar(x + 0.2, values, 0.38, label=model_name,
                          color=color, alpha=0.85, edgecolor='white')
            ax.bar_label(bars, fmt='%.4f', fontsize=8, rotation=90, padding=4)

    ax.set_xticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=9)
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel(metric)
    ax.set_ylim(0, ax.get_ylim()[1] * 1.2)
    ax.legend(fontsize=10)
    if metric == 'F1 Macro':
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x:.2f}'))
    else:
        ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))

plt.suptitle('Perbandingan Teknik Penanganan Class Imbalance (Validation Set)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT / 'nb05_technique_comparison.png', bbox_inches='tight', dpi=150)
plt.show()
print('Gambar disimpan.')

In [ ]:

# F1 per kelas: Baseline vs teknik terbaik (ambil dari per_class_report)
def f1_per_class_from_result(res: dict) -> np.ndarray:
    """Ekstrak F1 per kelas dari dict hasil evaluate()."""
    report = res.get('per_class_report', {})
    return np.array([report.get(cn, {}).get('f1-score', 0.0) for cn in class_names])

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

for ax, model_name in zip(axes, ['XGBoost', 'LightGBM']):
    # Kumpulkan semua teknik untuk model ini
    candidates = {
        'Baseline':        baseline_results[model_name],
        'ROS':             ros_results[model_name],
        'SMOTE':           smote_results[model_name],
        'Thr Calib':       threshold_results[model_name]['val'],
        **{f'W:{s}': weight_results[f'{model_name}_{s}'] for s in weight_schemes},
    }
    best_key = max(candidates, key=lambda k: candidates[k]['f1_macro'])
    best_res = candidates[best_key]

    f1_baseline = f1_per_class_from_result(baseline_results[model_name])
    f1_best     = f1_per_class_from_result(best_res)

    x = np.arange(n_classes)
    w = 0.38
    ax.bar(x - w/2, f1_baseline, w, label='Baseline',
           color='#4472C4', alpha=0.75, edgecolor='white')
    ax.bar(x + w/2, f1_best, w, label=f'Teknik terbaik: {best_key}',
           color='#ED7D31', alpha=0.85, edgecolor='white')

    # Tandai kelas langka
    for i, cnt in enumerate(counts):
        if cnt < 20:
            ax.axvspan(i - 0.5, i + 0.5, alpha=0.08, color='red', zorder=0)

    ax.set_xticks(x)
    ax.set_xticklabels(class_names, rotation=70, fontsize=8)
    ax.set_title(
        f'{model_name}: F1 per Kelas – Baseline vs {best_key}  '
        f'(F1m Baseline={baseline_results[model_name]["f1_macro"]:.4f} → '
        f'{best_res["f1_macro"]:.4f})',
        fontweight='bold', fontsize=10
    )
    ax.set_ylabel('F1 Score')
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=10)
    ax.text(0.01, 0.94, 'Latar merah = kelas sangat langka (< 20 sampel)',
            transform=ax.transAxes, fontsize=8, color='#B71C1C', va='top')

plt.suptitle('F1 per Kelas: Baseline vs Teknik Penanganan Imbalance Terbaik',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUT / 'nb05_f1_perclass_comparison.png', bbox_inches='tight', dpi=150)
plt.show()


---
## 8. Ringkasan

In [ ]:
# ── Ringkasan Notebook 5 (otomatis) ───────────────────────────────────
import json

best_f1m_row = max(rows, key=lambda r: r['F1 Macro'])
best_acc_row = max(rows, key=lambda r: r['Accuracy'])

# Hitung delta F1 Macro tiap teknik vs baseline per model
teknik_labels = ['Baseline', 'RandomOversample', 'SMOTE', 'best weight', 'Threshold Calib']

print('=' * 62)
print('  RINGKASAN NOTEBOOK 5 – Penanganan Class Imbalance')
print('=' * 62)
print(f'\nData:')
print(f'  Rasio imbalance          : {ratio:.0f}×  ({class_names[sort_idx[0]]} vs {min_cls})')
print(f'  Kelas sangat langka (< 20): {n_rare}')
print(f'  Training set setelah ROS : {len(y_ros):,}  ({len(y_ros)/len(y_train)*100:.0f}% dari baseline)')
print(f'  Training set setelah SMOTE: {len(y_smote):,}')
print()
print('Hasil terbaik per metrik (Validation Set):')
print(f'  F1 Macro terbaik : {best_f1m_row["Teknik"]} → {best_f1m_row["F1 Macro"]:.4f}')
print(f'  Accuracy terbaik : {best_acc_row["Teknik"]} → {best_acc_row["Accuracy"]:.4f}')
print()
print('Delta F1 Macro vs Baseline (rata-rata XGBoost + LightGBM):')
for t_key, t_label in [('RandomOversample', 'RandomOversample'),
                        ('SMOTE', 'SMOTE'),
                        ('best weight', 'Custom Weights (best)'),
                        ('Threshold Calib', 'Threshold Calibration')]:
    deltas = []
    for model_name in ['XGBoost', 'LightGBM']:
        bl_f1 = baseline_results[model_name]['f1_macro']
        tech_rows = [r for r in rows if r['Model'] == model_name and t_key in r['Teknik']]
        if tech_rows:
            deltas.append(max(tech_rows, key=lambda r: r['F1 Macro'])['F1 Macro'] - bl_f1)
    if deltas:
        avg_delta = np.mean(deltas)
        sign = '+' if avg_delta >= 0 else ''
        print(f'  {t_label:<30}: {sign}{avg_delta:.4f}')
print()
print('Output eksperimen:')
print(f'  Folder : experiments/2026-07-01/')
print(f'  CSV    : nb05_comparison.csv')
print(f'  Plot   : nb05_*.png (3 file)')
print(f'  Model  : *_ros.pkl, *_smote.pkl, *_thresholds.npy')
print()

# Simpan ringkasan ke JSON
summary_exp = {
    'date': '2026-07-01',
    'imbalance_ratio': float(ratio),
    'n_rare_classes': int(n_rare),
    'n_total_train': int(len(y_train)),
    'n_after_ros': int(len(y_ros)),
    'n_after_smote': int(len(y_smote)),
    'results': rows,
    'best_f1_macro': best_f1m_row,
    'best_accuracy': best_acc_row,
}
(OUT / 'nb05_summary.json').write_text(json.dumps(summary_exp, indent=2))
print('Ringkasan JSON disimpan: nb05_summary.json')